# 04. Rust Focused STAMPS Parity And Native Speed Evidence

This notebook captures the current Rust/native backend evidence without overstating the full parity gate.

Read it after `start_here.ipynb` and `03_pystamps_verification.ipynb`. This notebook answers four narrower questions:

- is the compiled Rust module importable, and which native exports are present;
- which passing MATLAB StaMPS golden-dataset audit is used as the focused parity evidence;
- do explicit `pystamps verify` reruns pass for each recorded `run_root` and `golden_root` pair;
- are the Rust-backed stage 4, 7, and 8 kernels faster than the Python implementations while matching numerically.

This notebook intentionally does not claim the full four-dataset manifest gate is green. It uses the clean focused small-baseline stage-7 audit artifact and does not claim the full-manifest gate.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import statistics
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from pystamps.kernels import (
    describe_backend_matrix,
    run_stage4_edge_stats_kernel,
    run_stage7_scla_kernel,
    run_stage8_edge_noise_kernel,
)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'pystamps').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from notebook execution directory')


REPO_ROOT = find_repo_root()
FOCUSED_AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_small_baseline_audit.json'
AUDITED_MANIFEST = REPO_ROOT / 'pystamps' / 'data' / 'audited_workflow_manifest.json'
DATASET_STAGE7_DIAG = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_small_baseline_stage7diag'
DATASET_STAGE7_FULL = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_small_baseline_stage7'

BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}



def failure_count(value: object) -> int:
    if isinstance(value, list):
        return len(value)
    if isinstance(value, int):
        return value
    return 0


def _ts(path: Path | None) -> str:
    if path is None or not path.exists():
        return '<missing>'
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat()


def run_command(cmd: list[str], *, check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    t0 = time.perf_counter()
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env or BASE_ENV,
        text=True,
        capture_output=True,
        check=False,
    )
    completed.elapsed_sec = time.perf_counter() - t0
    print('$', ' '.join(cmd))
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'command failed with exit code {completed.returncode}')
    return completed


native_import_error = None
native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
try:
    native_mod = importlib.import_module('pystamps.kernels._stage2_native') if native_spec is not None else None
except Exception as exc:
    native_import_error = f'{type(exc).__name__}: {exc}'
    native_mod = None
native_path = Path(native_spec.origin) if native_spec is not None and native_spec.origin and native_mod is not None else None
src_path = REPO_ROOT / 'src' / 'lib.rs'

artifact_rows = [
    {'artifact': 'rust_source', 'path': str(src_path), 'modified_utc': _ts(src_path)},
    {'artifact': 'focused_audit_json', 'path': str(FOCUSED_AUDIT_JSON), 'modified_utc': _ts(FOCUSED_AUDIT_JSON)},
    {'artifact': 'audited_workflow_manifest', 'path': str(AUDITED_MANIFEST), 'modified_utc': _ts(AUDITED_MANIFEST)},
]
if native_path is None:
    artifact_rows.append({'artifact': 'native_module', 'path': '<not importable>', 'modified_utc': '<missing>'})
else:
    artifact_rows.insert(0, {'artifact': 'native_module', 'path': str(native_path), 'modified_utc': _ts(native_path)})

pd.DataFrame(artifact_rows)


,artifact,path,modified_utc
0,native_module,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-28T07:40:10.115141+00:00
1,rust_source,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-14T11:47:54.153209+00:00
2,focused_audit_json,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-22T15:21:37.689397+00:00
3,audited_workflow_manifest,/shared/home/rdelprete/PythonProjects/AgenticW...,2026-04-22T14:52:21.097418+00:00


In [2]:
backend_matrix = describe_backend_matrix()
native_exports = sorted(name for name in dir(native_mod) if not name.startswith('_')) if native_mod is not None else []
speed_claim_kernels = ['stage4_edge_stats', 'stage7_scla', 'stage8_edge_noise']

pd.Series(
    {
        'python_executable': sys.executable,
        'python_version': sys.version.split()[0],
        'native_available': native_mod is not None,
        'native_origin': str(native_path) if native_path is not None else '<not importable>',
        'native_exports': ', '.join(native_exports) if native_exports else '<none>',
        'speed_claim_kernels': ', '.join(speed_claim_kernels),
        'stage4_native_available': 'native' in backend_matrix.get('kernels', {}).get('stage4_edge_stats', {}).get('available_backends', []),
        'stage7_native_available': 'native' in backend_matrix.get('kernels', {}).get('stage7_scla', {}).get('available_backends', []),
        'stage8_native_available': 'native' in backend_matrix.get('kernels', {}).get('stage8_edge_noise', {}).get('available_backends', []),
        'stage2_topofit_speed_claimed': False,
        'stage2_topofit_note': 'registered native path currently preserves parity by falling back to Python; not benchmarked here',
        'focused_audit_json_present': FOCUSED_AUDIT_JSON.exists(),
        'stage7_diag_dataset_present': DATASET_STAGE7_DIAG.exists(),
        'stage7_full_dataset_present': DATASET_STAGE7_FULL.exists(),
        'native_import_error': native_import_error or '<none>',
    }
)


python_executable               /shared/home/rdelprete/PythonProjects/AgenticW...
python_version                                                             3.14.2
native_available                                                             True
native_origin                   /shared/home/rdelprete/PythonProjects/AgenticW...
native_exports                  accumulate_weighted_grid, histogram_with_cente...
speed_claim_kernels             stage4_edge_stats, stage7_scla, stage8_edge_noise
stage4_native_available                                                      True
stage7_native_available                                                      True
stage8_native_available                                                      True
stage2_topofit_speed_claimed                                                False
stage2_topofit_note             registered native path currently preserves par...
focused_audit_json_present                                                   True
stage7_diag_data

## Focused MATLAB StaMPS parity audit artifact

This notebook uses the passing focused small-baseline stage-7 audit artifact at `inputs_and_outputs/validation_runs/latest_small_baseline_audit.json`.

That artifact compares regenerated pySTAMPS outputs against the trusted MATLAB StaMPS golden dataset roots recorded in the payload. It is intentionally narrower than the full manifest gate; use `make audit` for that separate full-gate claim.

To refresh this focused artifact:

```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 PYTHONPATH=. \
  uv run python scripts/validate_audit.py \
    --allow-subset \
    --datasets inputs_and_outputs/InSAR_dataset_small_baseline_stage7diag inputs_and_outputs/InSAR_dataset_small_baseline_stage7 \
    --output inputs_and_outputs/validation_runs/latest_small_baseline_audit.json
```

The cells below first display the recorded audit rows, then rerun `pystamps verify` against each `run_root` / `golden_root` pair.


In [3]:
if not FOCUSED_AUDIT_JSON.exists():
    raise FileNotFoundError(f'Focused audit artifact is missing: {FOCUSED_AUDIT_JSON}')

audit_payload = json.loads(FOCUSED_AUDIT_JSON.read_text(encoding='utf-8'))
audit_rows = audit_payload.get('audits', [])
manifest_payload = json.loads(AUDITED_MANIFEST.read_text(encoding='utf-8'))
manifest_targets = {
    Path(target['local_dataset_path']).name: target
    for target in manifest_payload.get('workflow_targets', [])
}

audit_summary = pd.DataFrame(
    [
        {
            'dataset': audit['dataset'],
            'workflow': audit.get('workflow', '<missing>'),
            'status': audit['status'],
            'ok': audit['ok'],
            'checked': audit['checked'],
            'failed_count': failure_count(audit.get('failed', [])),
            'run_root': audit['run_root'],
            'golden_root': audit['golden_root'],
        }
        for audit in audit_rows
    ]
)
display(audit_summary)

pd.Series(
    {
        'artifact': str(FOCUSED_AUDIT_JSON.relative_to(REPO_ROOT)),
        'generated_at_utc': audit_payload.get('generated_at_utc', '<missing>'),
        'completed': audit_payload.get('completed', False),
        'clean_completion': bool(audit_payload.get('completed') and not audit_payload.get('interrupted')),
        'audit_rows': len(audit_rows),
        'ok': audit_payload.get('ok', False),
        'failed_workflows': ', '.join(audit_payload.get('failed_workflows', [])) or '<none>',
        'scope': 'focused small-baseline stage-7 STAMPS golden dataset parity',
        'full_manifest_claimed': False,
    }
)


,dataset,workflow,status,ok,checked,failed_count,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,InSAR_dataset_small_baseline_stage7diag_audit,passed,True,4,0,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...
1,InSAR_dataset_small_baseline_stage7,InSAR_dataset_small_baseline_stage7_audit,passed,True,4,0,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...


artifact                 inputs_and_outputs/validation_runs/latest_smal...
generated_at_utc                                      2026-04-22T15:21:36Z
completed                                                             True
clean_completion                                                      True
audit_rows                                                               2
ok                                                                    True
failed_workflows                                                    <none>
scope                    focused small-baseline stage-7 STAMPS golden d...
full_manifest_claimed                                                False
dtype: object

In [4]:
oracle_rows = []
for audit in audit_rows:
    target = manifest_targets.get(audit['dataset'], {})
    oracle_rows.append(
        {
            'dataset': audit['dataset'],
            'manifest_id': target.get('id', '<missing>'),
            'kind': target.get('kind', '<missing>'),
            'audit_start_step': target.get('audit_start_step', '<default>'),
            'audit_end_step': target.get('audit_end_step', '<default>'),
            'oracle_reference_paths': ', '.join(target.get('oracle_reference_paths', [])) or '<dataset golden root>',
        }
    )
display(pd.DataFrame(oracle_rows))

verify_rows: list[dict[str, object]] = []
for audit in audit_rows:
    verify_cmd = [
        sys.executable,
        '-m',
        'pystamps.cli',
        'verify',
        '--run',
        str(audit['run_root']),
        '--golden',
        str(audit['golden_root']),
    ]
    completed = run_command(verify_cmd, check=False)
    verify_payload = json.loads(completed.stdout) if completed.stdout.strip().startswith('{') else {'ok': False, 'checked': 0, 'failed': []}
    verify_rows.append(
        {
            'dataset': audit['dataset'],
            'returncode': completed.returncode,
            'elapsed_sec': round(completed.elapsed_sec, 3),
            'ok': bool(verify_payload.get('ok')),
            'checked': int(verify_payload.get('checked', 0)),
            'failed_count': failure_count(verify_payload.get('failed', [])),
            'run_root': audit['run_root'],
            'golden_root': audit['golden_root'],
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df)


,dataset,manifest_id,kind,audit_start_step,audit_end_step,oracle_reference_paths
0,InSAR_dataset_small_baseline_stage7diag,small_baseline_diagnostic,small_baseline,7,7,"StaMPS/matlab/ps_calc_scla.m, StaMPS/matlab/ps..."
1,InSAR_dataset_small_baseline_stage7,small_baseline_full,small_baseline,7,7,"StaMPS/matlab/ps_calc_scla.m, StaMPS/matlab/ps..."


$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python -m pystamps.cli verify --run /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260422_152136/InSAR_dataset_small_baseline_stage7diag_stage7_7 --golden /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_small_baseline_stage7diag
{
  "ok": true,
  "checked": 4,
  "failed": []
}



$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python -m pystamps.cli verify --run /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260422_152136/InSAR_dataset_small_baseline_stage7_stage7_7 --golden /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_small_baseline_stage7
{
  "ok": true,
  "checked": 4,
  "failed": []
}



,dataset,returncode,elapsed_sec,ok,checked,failed_count,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,0,18.608,True,4,0,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...
1,InSAR_dataset_small_baseline_stage7,0,15.509,True,4,0,/shared/home/rdelprete/PythonProjects/AgenticW...,/shared/home/rdelprete/PythonProjects/AgenticW...


In [5]:
pytest_expr = (
    'stage4_stage7_stage8_native_kernels_match_python_reference '
    'or stage2_histogram_kernel_native_equal_spacing_matches_cpu'
)
if native_mod is None:
    pytest_completed = None
    pytest_result = pd.Series(
        {
            'returncode': '<skipped>',
            'selection': pytest_expr,
            'reason': 'native extension is not importable in this environment',
        }
    )
else:
    pytest_cmd = [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        'tests/test_kernels_accelerated.py',
        '-k',
        pytest_expr,
    ]
    pytest_completed = run_command(pytest_cmd, check=False)
    pytest_result = pd.Series(
        {
            'returncode': pytest_completed.returncode,
            'elapsed_sec': round(pytest_completed.elapsed_sec, 3),
            'selection': pytest_expr,
        }
    )
pytest_result


$ /shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/.venv/bin/python -m pytest -q tests/test_kernels_accelerated.py -k stage4_stage7_stage8_native_kernels_match_python_reference or stage2_histogram_kernel_native_equal_spacing_matches_cpu
..                                                                       [100%]



returncode                                                     0
elapsed_sec                                               23.546
selection      stage4_stage7_stage8_native_kernels_match_pyth...
dtype: object

In [6]:
verify_ok = bool(not verify_df.empty and verify_df['ok'].all() and (verify_df['returncode'] == 0).all())
audit_ok = bool(
    audit_payload.get('ok')
    and audit_payload.get('completed')
    and not audit_payload.get('interrupted')
    and len(audit_rows) == 2
)
parity_steps = pd.DataFrame(
    [
        {
            'step': '1. Load focused audit artifact',
            'evidence': str(FOCUSED_AUDIT_JSON.relative_to(REPO_ROOT)),
            'ok': audit_ok,
        },
        {
            'step': '2. Identify MATLAB StaMPS golden roots',
            'evidence': '; '.join(sorted(audit['golden_root'] for audit in audit_rows)),
            'ok': all(Path(audit['golden_root']).exists() for audit in audit_rows),
        },
        {
            'step': '3. Rerun pystamps verify for each audit row',
            'evidence': f"{int(verify_df['checked'].sum()) if not verify_df.empty else 0} files checked, {int(verify_df['failed_count'].sum()) if not verify_df.empty else 0} failures",
            'ok': verify_ok,
        },
        {
            'step': '4. Focused parity verdict',
            'evidence': 'small-baseline stage-7 pySTAMPS outputs match the recorded StaMPS golden datasets',
            'ok': audit_ok and verify_ok,
        },
    ]
)
display(parity_steps)


,step,evidence,ok
0,1. Load focused audit artifact,inputs_and_outputs/validation_runs/latest_smal...,True
1,2. Identify MATLAB StaMPS golden roots,/shared/home/rdelprete/PythonProjects/AgenticW...,True
2,3. Rerun pystamps verify for each audit row,"8 files checked, 0 failures",True
3,4. Focused parity verdict,small-baseline stage-7 pySTAMPS outputs match ...,True


## Rust speed evidence

This benchmark compares Python and compiled Rust/native kernels that actually dispatch to Rust in this checkout: stage 4 edge statistics, stage 7 SCLA, and stage 8 edge noise.

Stage-2 topofit is deliberately not used as speed evidence here. Its registered native path currently preserves parity by falling back to the Python reference path, so a topofit speed table would not be a valid Rust acceleration claim.

The synthetic inputs are deterministic and sized to make the speed difference visible without rerunning the full audit. The `max_abs` and `parity_ok` columns check numerical agreement for the measured outputs.


In [7]:
bench_rows: list[dict[str, object]] = []
speed_summary = pd.Series(
    {
        'benchmarked_kernels': '<not measured>',
        'min_speedup_vs_python': '<not measured>',
        'all_rust_faster': False,
        'all_parity_ok': False,
    }
)


def timed(fn: Callable[[], Any], repeats: int) -> tuple[list[float], Any]:
    fn()
    runs: list[float] = []
    output: Any = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        output = fn()
        runs.append(time.perf_counter() - t0)
    return runs, output


def max_abs_array(left: np.ndarray, right: np.ndarray) -> float:
    left_arr = np.asarray(left)
    right_arr = np.asarray(right)
    if np.array_equal(left_arr, right_arr, equal_nan=True):
        return 0.0
    finite = np.isfinite(left_arr) & np.isfinite(right_arr)
    if not np.all(np.equal(left_arr[~finite], right_arr[~finite])):
        return float('inf')
    return float(np.max(np.abs(left_arr[finite] - right_arr[finite]))) if np.any(finite) else 0.0


def max_abs_output(left: dict[str, np.ndarray], right: dict[str, np.ndarray]) -> float:
    keys = sorted(set(left) | set(right))
    if set(left) != set(right):
        return float('inf')
    return max(max_abs_array(np.asarray(left[key]), np.asarray(right[key])) for key in keys)


def record_kernel(
    kernel: str,
    python_fn: Callable[[], dict[str, np.ndarray]],
    native_fn: Callable[[], dict[str, np.ndarray]],
    tolerance: float,
    repeats: int,
) -> None:
    python_runs, python_output = timed(python_fn, repeats)
    native_runs, native_output = timed(native_fn, repeats)
    py_mean = statistics.mean(python_runs)
    native_mean = statistics.mean(native_runs)
    max_abs = max_abs_output(python_output, native_output)
    bench_rows.append(
        {
            'kernel': kernel,
            'python_mean_sec': py_mean,
            'native_mean_sec': native_mean,
            'speedup_vs_python': py_mean / native_mean if native_mean > 0 else float('inf'),
            'python_runs_sec': ', '.join(f'{value:.4f}' for value in python_runs),
            'native_runs_sec': ', '.join(f'{value:.4f}' for value in native_runs),
            'max_abs': max_abs,
            'tolerance': tolerance,
            'parity_ok': bool(max_abs <= tolerance),
            'rust_faster': bool(native_mean < py_mean),
        }
    )


if native_mod is None:
    bench_rows.append(
        {
            'kernel': '<native unavailable>',
            'parity_ok': False,
            'rust_faster': False,
            'reason': 'build/install the Rust extension before measuring native speed',
        }
    )
else:
    rng = np.random.default_rng(321)
    repeats = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '2'))

    n_ps4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_PS', '30000'))
    n_ifg4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_IFG', '32'))
    n_edge4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_EDGES', '60000'))
    ph_weed = np.exp(1j * rng.normal(size=(n_ps4, n_ifg4))).astype(np.complex128)
    node_a4 = rng.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    node_b4 = rng.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    bperp4 = np.linspace(-80.0, 120.0, n_ifg4, dtype=np.float64)
    day4 = np.linspace(0.0, 365.0, n_ifg4, dtype=np.float64)
    record_kernel(
        'stage4_edge_stats',
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='python'),
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='native'),
        1.0e-10,
        repeats,
    )

    n_ps7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_PS', '30000'))
    n_ifg7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_IFG', '36'))
    ph_proc = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    ph_mean_v = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    bperp_mat = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    unwrap_ix = np.arange(n_ifg7, dtype=np.int64)
    solve_ix = np.arange(n_ifg7, dtype=np.int64)
    day7 = np.linspace(0.0, 365.0, n_ifg7, dtype=np.float64)
    ifg_std = np.full(n_ifg7, 10.0, dtype=np.float64)
    record_kernel(
        'stage7_scla',
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='python'),
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='native'),
        1.0e-4,
        repeats,
    )

    n_ps8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_PS', '50000'))
    n_ifg8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_IFG', '36'))
    n_edge8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_EDGES', '120000'))
    uw_ph = np.exp(1j * rng.normal(size=(n_ps8, n_ifg8))).astype(np.complex64)
    node_a8 = rng.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    node_b8 = rng.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    record_kernel(
        'stage8_edge_noise',
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='python'),
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='native'),
        1.0e-5,
        repeats,
    )

bench_df = pd.DataFrame(bench_rows)
if not bench_df.empty and 'speedup_vs_python' in bench_df:
    speed_summary = pd.Series(
        {
            'benchmarked_kernels': ', '.join(bench_df['kernel'].astype(str)),
            'min_speedup_vs_python': round(float(bench_df['speedup_vs_python'].min()), 3),
            'all_rust_faster': bool(bench_df['rust_faster'].all()),
            'all_parity_ok': bool(bench_df['parity_ok'].all()),
            'repeats': int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '2')),
        }
    )

display(bench_df)
speed_summary


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,python_runs_sec,native_runs_sec,max_abs,tolerance,parity_ok,rust_faster
0,stage4_edge_stats,4.850234,0.747142,6.491714,"4.8724, 4.8280","0.8241, 0.6702",2.220446e-15,1.000000e-10,True,True
1,stage7_scla,0.097796,0.022355,4.374692,"0.1144, 0.0812","0.0218, 0.0229",3.051758e-05,1.000000e-04,True,True
2,stage8_edge_noise,0.098791,0.034537,2.860436,"0.1038, 0.0938","0.0338, 0.0353",4.768372e-07,1.000000e-05,True,True


benchmarked_kernels      stage4_edge_stats, stage7_scla, stage8_edge_noise
min_speedup_vs_python                                                 2.86
all_rust_faster                                                       True
all_parity_ok                                                         True
repeats                                                                  2
dtype: object

In [8]:
summary = pd.Series(
    {
        'focused_audit_ok': audit_ok,
        'focused_verify_ok': verify_ok,
        'pytest_ok': False if pytest_completed is None else pytest_completed.returncode == 0,
        'speed_benchmark_ok': bool(
            not bench_df.empty
            and 'parity_ok' in bench_df
            and bench_df['parity_ok'].all()
            and bench_df['rust_faster'].all()
        ),
        'min_native_speedup_vs_python': speed_summary.get('min_speedup_vs_python', '<not measured>'),
        'benchmarked_native_kernels': speed_summary.get('benchmarked_kernels', '<not measured>'),
        'full_manifest_claimed': False,
    }
)
display(summary)


focused_audit_ok                                                             True
focused_verify_ok                                                            True
pytest_ok                                                                    True
speed_benchmark_ok                                                           True
min_native_speedup_vs_python                                                 2.86
benchmarked_native_kernels      stage4_edge_stats, stage7_scla, stage8_edge_noise
full_manifest_claimed                                                       False
dtype: object

## What to do with the result

- If the focused audit, explicit verify reruns, pytest smoke, and stage4/stage7/stage8 speed checks are green, this notebook supports focused small-baseline stage-7 parity against the recorded MATLAB StaMPS golden datasets plus native Rust speedups for the measured kernels.
- Do not use this notebook to claim full four-dataset manifest parity; use `make audit` for that gate.
- Do not use stage-2 topofit timings as Rust speed evidence until the registered native topofit path stops falling back to Python for parity.
